# Luma Phase 2 Module Implementation
This notebook contain the implementation of phase 2 of Luma Geospatial Engine

## Prerequisite
Earth engine initialization using service account

In [6]:
import ee 
import luma_ge
#Manuall Authentication, use personal account (uncomment if needed)
luma_ge.authenticate_manually()
#Autheticate using service account (json file)
#use relevant file in auth folder
#service_account_path = '../auth/epistem-luma-launching-92541db9962d.json'
#success = luma_ge.initialize_with_service_account(service_account_path)
#Check authentication status
status = luma_ge.get_auth_status()
print(f"Initialized: {status['initialized']}")
print(f"Authenticated: {status['authenticated']}")
if status['project']:
    print(f"Project: {status['project']}")

2026-11-05 10:16:47,637 - luma_ge.ee_config - INFO - Starting manual Earth Engine authentication...
2026-11-05 10:16:50,684 - luma_ge.ee_config - INFO - Earth Engine authenticated and initialized successfully


Initialized: True
Authenticated: True
Project: projects/161874247127/assets/Final_GT_Clean_TL_2025


## Retrieve AOI from Earth Engine Asset Manager

In [7]:
from luma_ge.helpers import GEE_Asset_Manager
#Initialize the Asset Manager from Earth Engine
asset = GEE_Asset_Manager()
if asset.load_asset():
    regency_names = asset.get_regency_names()
    if regency_names:
        print(f"✓ Found {len(regency_names)} regencies")
        #use city name
        selected_regency = "Kota Bandung" 
        #Verify the city name exists in the regency list
        if selected_regency in regency_names:
            print(f"\n✓ Selected regency: {selected_regency}")
        else:
            print(f"\n✗ Regency '{selected_regency}' not found")
            print(f"\nAvailable regencies:")
            for i, name in enumerate(regency_names, 1):
                print(f"  {i}. {name}")
            selected_regency = None
    else:
        print("✗ No regency names found")
        selected_regency = None
else:
    print("✗ Failed to load asset")
    selected_regency = None
#Retrieve AOI geometry for the selected regency
if selected_regency:
    aoi = asset.get_regency_geometry(selected_regency)
    if aoi:
        print(f"✓ Successfully retrieved geometry for: {selected_regency}")
    else:
        print(f"✗ Failed to retrieve geometry for: {selected_regency}")
        aoi = None
else:
    print("✗ No regency selected")
    aoi = None

2026-11-05 10:17:03,843 - luma_ge.helpers - INFO - Successfully loaded asset with 548 features
2026-11-05 10:17:05,209 - luma_ge.helpers - INFO - Extracted 516 unique regency names


✓ Found 516 regencies

✓ Selected regency: Kota Bandung


2026-11-05 10:17:06,025 - luma_ge.helpers - INFO - Successfully retrieved geometry for: Kota Bandung


✓ Successfully retrieved geometry for: Kota Bandung


## New Feature: Sentinel-2 Data Retrieval
Note: Sentinel 2 data has varying spatial resolution. Use Sharpen=True, if spatial resolution harmonization is desired.
The approach downscale the 20m bands into 10m band. The 60m band will be removed regardless of the harmonization

In [8]:
from luma_ge.data_acquisition import Reflectance_Data, final_Image
import geemap
#aoi = geemap.shp_to_ee('../data/area_of_interest.shp')
#initilizate the class
optical_reflectance = Reflectance_Data()
composite = final_Image()
#define the temporal range
start = '2019-01-01'
end = '2019-12-30'
#retrieve the sentinel 2 data image collection
s2_coll, metas2 = optical_reflectance.get_s2_optical_data(aoi, start, end, cloud_cover=40, cs_threshold = 0.6, compute_detailed_stats=False)
#composite the collection. image sharpening is optional, if not applied, use sharpen=False
s2_10m = composite.get_temporal_composite(s2_coll, aoi, calculate_coverage=False, coverage_scale=10, sharpen=True)
#select the sharrpened image from the composite result, and applied the scale factors to convert to reflectance
raw_s2_10m = s2_10m['image']  #Still in DN (0–10000)
scaled_s2 = optical_reflectance.apply_s2_scale_factors(raw_s2_10m )
#Visualization for 10m band
vis_10m = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['RED', 'GREEN', 'BLUE']}
#Visualization for 20m band
res_20m = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['RED_EDGE1', 'RED_EDGE4', 'SWIR1']}
Map = geemap.Map()
Map.addLayer(scaled_s2, vis_10m, 'Sentinel-2 Composite')
Map.centerObject(aoi, 10)
Map

2026-11-05 10:17:07,155 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-11-05 10:17:07,156 - final_Image - INFO - final_Image creation initialized.
2026-11-05 10:17:07,158 - Reflectance_Data - INFO - Date range: 2019-01-01 to 2019-12-30
2026-11-05 10:17:07,159 - Reflectance_Data - INFO - Cloud cover threshold (image-level): 40%
2026-11-05 10:17:07,159 - Reflectance_Data - INFO - Cloud Score+ pixel threshold: 0.6
2026-11-05 10:17:07,160 - Reflectance_Data - INFO - Detailed statistics will not be computed
2026-11-05 10:17:07,162 - Reflectance_Data - INFO - Using Sentinel-2 Level 2A SR data.
2026-11-05 10:17:07,164 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-11-05 10:17:07,165 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)
2026-11-05 10:17:08,137 - final_Image - INFO - Creating median composite from 156 images
2026-11-05 10:17:08,138 - final_Image - INFO - Composite clipped to AOI
2026-11-05

Map(center=[-6.919241950180151, 107.63659926544979], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
export_band = scaled_s2.select('RED_EDGE1','RED_EDGE2', 'RED_EDGE3', 'RED_EDGE4', 'SWIR1', 'SWIR2')
export_task = ee.batch.Export.image.toDrive(
     image=export_band,
     description='S2_SharpenLuma_v1',
     folder='Earth Engine',
     fileNamePrefix='S2_SharpenLuma_v1',
     scale=10,
     region=aoi,  # or aoi.geometry()
     maxPixels=1e13
 )
export_task.start()
import time

while export_task.active():
     print('Exporting... (status: {})'.format(export_task.status()['state']))
     time.sleep(10)

print('Export complete (status: {})'.format(export_task.status()['state']))

## Topographic Correction

In [10]:
from luma_ge.imagery_correction import Topographic_Correction
topographic_correction = Topographic_Correction()
#Apply SCSc topographic correction before compositing (10 m for Sentinel-2).
#NASADEM is used by default; pass dem=custom_dem to use another elevation model.
# Option A: Default 6 bands (BLUE, GREEN, RED, NIR, SWIR1, SWIR2)
#corrected_s2_data = topographic_correction.correct_collection(s2_coll, scale=10)
#Option B (Optional): Correct all 10 Sentinel-2 bands
corrected_s2_data = topographic_correction.correct_collection(
     s2_coll,
     scale=10,
     band_list=Topographic_Correction.S2_OPTICAL_BANDS, )
comp_corrected = composite.get_temporal_composite(
    corrected_s2_data,
    aoi,
    reducer='median',
    calculate_coverage=False,
    coverage_scale=10,
    sharpen=True,
)
sharpen_corrected = comp_corrected['image']
scaled_s2 = optical_reflectance.apply_s2_scale_factors(sharpen_corrected)
Map.addLayer(scaled_s2 , vis_10m, 'Topographically Corrected S2 (10m RGB - DN)')
Map

2026-11-05 10:20:59,021 - Topographic_Correction - INFO - Topographic_Correction initialised.
2026-11-05 10:20:59,023 - Topographic_Correction - INFO - Using default DEM: NASA/NASADEM_HGT/001 (band: elevation)
2026-11-05 10:20:59,024 - Topographic_Correction - INFO - Applying SCSc topographic correction to collection (scale=10m, slope_threshold=5.0 deg)
2026-11-05 10:20:59,050 - Topographic_Correction - INFO - SCSc correction mapped over collection.
2026-11-05 10:20:59,831 - final_Image - INFO - Creating median composite from 156 images
2026-11-05 10:20:59,833 - final_Image - INFO - Composite clipped to AOI
2026-11-05 10:21:01,185 - googleapiclient.http - WARNING - Sleeping 1.49 seconds before retry 1 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/161874247127/value:compute?prettyPrint=false&alt=json, after 429
2026-11-05 10:21:05,060 - googleapiclient.http - WARNING - Sleeping 3.93 seconds before retry 2 of 5 for request: POST https://earthengine.googleapis.com/

EEException: Too many concurrent aggregations.